# Capítulo 6 · El cliente y los transportes

El servidor de la Clínica Veterinaria Naranco con tres transportes: en memoria, por stdio y por HTTP. Se mide lo que cuesta cada uno, qué pasa cuando dos recepcionistas reservan a la vez y qué pasa cuando dos servidores publican una herramienta con el mismo nombre.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. Las dos primeras partes no la necesitan; la tercera, sí.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Los datos y el servidor de la clínica

El servidor es un programa aparte; se guarda en un archivo para poder lanzarlo como un proceso.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 6: el cliente y los transportes. El servidor de la clínica en memoria, por
stdio y por HTTP; dos recepcionistas a la vez; y dos herramientas con el mismo nombre."""
import asyncio
import json
import re
import time
import subprocess
import sys
import urllib.error
import urllib.request
from contextlib import AsyncExitStack
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client, StdioServerParameters
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
MODELO = None                               # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/"
HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo

In [ ]:
import os
os.makedirs("datos", exist_ok=True)
for archivo in ("mascotas", "duenos", "historial", "vacunas", "citas", "farmacia", "facturas",
                "veterinarios", "tarifas"):
    if not os.path.exists(f"datos/{archivo}.json"):
        urllib.request.urlretrieve(f"{REPOSITORIO}datos/{archivo}.json", f"datos/{archivo}.json")

In [ ]:
%%writefile servidor_clinica.py
"""Servidor MCP de la Clínica Veterinaria Naranco: una versión reducida del capítulo 5, como
programa aparte, con las herramientas de consulta y de agenda y una más, citas_pendientes.

Se puede usar de tres formas: importarlo y conectar un cliente en memoria, lanzarlo como un
proceso que habla por stdio o lanzarlo como un servicio HTTP al que se conectan varios
clientes a la vez.
Uso: python servidor_clinica.py [carpeta_de_datos] [--http PUERTO]
"""
import argparse
import copy
import json
from datetime import date
from pathlib import Path
from typing import Annotated

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field

HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


# --- Los datos de la clínica ----------------------------------------------------

def cargar_datos(carpeta="datos"):
    global MASCOTAS, DUENOS, HISTORIAL, VACUNAS, CITAS0, FARMACIA, FACTURAS, VETERINARIOS, HORAS
    leer = lambda archivo: json.loads((Path(carpeta) / archivo).read_text(encoding="utf-8"))
    MASCOTAS, DUENOS = leer("mascotas.json"), leer("duenos.json")
    HISTORIAL, VACUNAS = leer("historial.json"), leer("vacunas.json")
    CITAS0, FARMACIA = leer("citas.json"), leer("farmacia.json")
    FACTURAS, VETERINARIOS = leer("facturas.json"), leer("veterinarios.json")
    HORAS = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reiniciar()


def reiniciar():
    """Deja la clínica como al principio: la agenda original y ningún aviso."""
    global CITAS, AVISOS
    CITAS, AVISOS = copy.deepcopy(CITAS0), []


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar nombres escritos de cualquier forma."""
    return str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu")).strip()


def edad(nacimiento):
    n = date.fromisoformat(nacimiento)
    return HOY.year - n.year - ((HOY.month, HOY.day) < (n.month, n.day))


def resumen_mascota(mid):
    m = MASCOTAS[mid]
    return {"nombre": m["nombre"], "especie": m["especie"], "raza": m["raza"],
            "edad_anos": edad(m["nacimiento"]), "peso_actual_kg": m["peso_kg"],
            "alergias": m["alergias"] or "ninguna conocida", "dueno": DUENOS[m["dueno"]]["nombre"]}


def veterinario_de(nombre):
    for v, n in VETERINARIOS.items():
        if sin_tildes(nombre) in (sin_tildes(v), sin_tildes(n)) or sin_tildes(nombre) in sin_tildes(n):
            return v
    return None


def mascota_de(nombre, dueno=""):
    ids = [m for m, d in MASCOTAS.items() if sin_tildes(nombre) == sin_tildes(d["nombre"])
           and sin_tildes(dueno) in sin_tildes(DUENOS[d["dueno"]]["nombre"])]
    if not ids:
        raise ToolError(f"No hay ninguna mascota llamada «{nombre}»"
                        + (f" de {dueno}" if dueno else "") + " en la clínica.")
    if len(ids) > 1:
        raise ToolError(f"Hay {len(ids)} mascotas llamadas «{nombre}»; indica el nombre del dueño.")
    return ids[0]


def libres(fecha, veterinario=""):
    try:
        dia = date.fromisoformat(fecha)
    except ValueError:
        raise ToolError(f"«{fecha}» no es una fecha; usa AAAA-MM-DD.")
    if dia.weekday() >= 5:
        raise ToolError(f"El {fecha} es fin de semana y la clínica cierra.")
    vet = veterinario_de(veterinario) if veterinario else None
    if veterinario and vet is None:
        raise ToolError(f"No hay ningún veterinario «{veterinario}». Son: "
                        + ", ".join(VETERINARIOS.values()) + ".")
    ocupadas = {(c["veterinario"], c["hora"]) for c in CITAS
                if c["fecha"] == fecha and c["estado"] != "cancelada"}
    return {"fecha": fecha, "horas_libres": {VETERINARIOS[v]: [h for h in HORAS if (v, h) not in ocupadas]
                                             for v in ([vet] if vet else VETERINARIOS)}}


# --- El servidor ------------------------------------------------------------------

Nombre = Annotated[str, Field(description="Nombre de la mascota.")]
Dueno = Annotated[str, Field(description="Nombre del dueño; solo si hay varias mascotas con ese nombre.")]
Fecha = Annotated[str, Field(description="AAAA-MM-DD.")]


def crear_servidor():
    servidor = MCPServer("clinica", instructions="Mascotas, vacunas, historial y agenda de la Clínica "
                         "Veterinaria Naranco.")

    @servidor.tool()
    def buscar_mascota(nombre: Nombre, dueno: Dueno = "") -> dict:
        """Datos de una mascota buscada por su nombre: especie, edad, peso, alergias y dueño."""
        return resumen_mascota(mascota_de(nombre, dueno))

    @servidor.tool()
    def vacunas(mascota: Nombre, dueno: Dueno = "") -> dict:
        """Vacunas de una mascota: la próxima fecha de cada una y si está vencida."""
        mid = mascota_de(mascota, dueno)
        return {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "vacunas": [
            {"vacuna": v["vacuna"], "proxima": v["proxima"],
             "estado": "vencida" if v["proxima"] < HOY.isoformat() else "al día"} for v in VACUNAS[mid]]}

    @servidor.tool()
    def historial(mascota: Nombre, ultimas: Annotated[int, Field(
            description="Cuántas visitas, de la más reciente a la más antigua.")] = 3,
                  dueno: Dueno = "") -> dict:
        """Las últimas visitas de una mascota, empezando por la más reciente."""
        mid = mascota_de(mascota, dueno)
        return {"mascota": MASCOTAS[mid]["nombre"], "visitas": [
            {k: v[k] for k in ("fecha", "motivo", "tratamiento", "peso_kg")}
            for v in reversed(HISTORIAL[mid][-ultimas:])]}

    @servidor.tool()
    def huecos_libres(fecha: Fecha, veterinario: Annotated[str, Field(
            description="Nombre del veterinario; vacío para todos.")] = "") -> dict:
        """Horas libres de la agenda en un día laborable, de un veterinario o de todos."""
        return libres(fecha, veterinario)

    @servidor.tool()
    def citas_de(mascota: Nombre, dueno: Dueno = "") -> dict:
        """Citas de una mascota (pasadas y pendientes), con su id."""
        mid = mascota_de(mascota, dueno)
        return {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "citas": [
            {"id": c["id"], "fecha": c["fecha"], "hora": c["hora"], "motivo": c["motivo"],
             "veterinario": VETERINARIOS[c["veterinario"]], "estado": c["estado"]}
            for c in CITAS if c["mascota"] == mid]}

    @servidor.tool()
    def reservar_cita(mascota: Nombre,
                      veterinario: Annotated[str, Field(description="Nombre del veterinario.")],
                      fecha: Fecha, hora: Annotated[str, Field(description="HH:MM, en punto o y media.")],
                      motivo: Annotated[str, Field(description="Motivo de la cita.")],
                      dueno: Dueno = "") -> dict:
        """Reserva una cita en una hora libre."""
        mid, vet = mascota_de(mascota, dueno), veterinario_de(veterinario)
        if vet is None:
            raise ToolError(f"No hay ningún veterinario «{veterinario}».")
        libres(fecha, veterinario)
        if hora not in HORAS:
            raise ToolError(f"«{hora}» no es una hora de consulta.")
        if any(c["fecha"] == fecha and c["hora"] == hora and c["veterinario"] == vet
               and c["estado"] != "cancelada" for c in CITAS):
            raise ToolError(f"{VETERINARIOS[vet]} ya tiene una cita el {fecha} a las {hora}.")
        cita = {"id": f"CT-{len(CITAS0) + len(CITAS) + 1:04d}", "fecha": fecha, "hora": hora,
                "veterinario": vet, "mascota": mid, "motivo": motivo, "estado": "pendiente"}
        CITAS.append(cita)
        return {"reservada": cita["id"], "mascota": MASCOTAS[mid]["nombre"],
                "veterinario": VETERINARIOS[vet], "fecha": fecha, "hora": hora}

    @servidor.tool()
    def cancelar_cita(cita: Annotated[str, Field(description="Id de la cita, como CT-0026 "
                                                             "(lo da citas_de).")]) -> dict:
        """Cancela una cita pendiente."""
        for c in CITAS:
            if c["id"] == cita:
                if c["estado"] != "pendiente":
                    raise ToolError(f"La cita {cita} está «{c['estado']}».")
                c["estado"] = "cancelada"
                return {"cancelada": cita, "fecha": c["fecha"], "hora": c["hora"]}
        raise ToolError(f"No existe la cita {cita}.")

    @servidor.tool()
    def citas_pendientes() -> dict:
        """Todas las citas pendientes de la agenda, para comprobar qué se ha reservado."""
        return {"citas": [{k: c[k] for k in ("id", "fecha", "hora", "veterinario", "mascota")}
                          for c in CITAS if c["estado"] == "pendiente"]}

    return servidor


if __name__ == "__main__":
    argumentos = argparse.ArgumentParser()
    argumentos.add_argument("datos", nargs="?", default="datos")
    argumentos.add_argument("--http", type=int, help="puerto; sin él, el servidor habla por stdio")
    a = argumentos.parse_args()
    cargar_datos(a.datos)
    servidor = crear_servidor()
    if a.http:
        servidor.run(transport="streamable-http", host="127.0.0.1", port=a.http)
    else:
        servidor.run()

In [ ]:
SERVIDOR = Path("servidor_clinica.py")      # en el cuaderno, en la carpeta de trabajo
sys.path.insert(0, str(SERVIDOR.parent))
import servidor_clinica  # noqa: E402

PUERTO = 8765

## 2. Lo que cuesta cada transporte

In [ ]:
def en_memoria():
    """Un servidor en el mismo programa: todos los clientes comparten sus datos."""
    return servidor_clinica.crear_servidor()


def por_stdio():
    """Cada cliente lanza su propio proceso del servidor, con su propia copia de los datos."""
    return StdioServerParameters(command=sys.executable, args=[str(SERVIDOR), str(DATOS)])


def por_http():
    """La dirección de un servidor HTTP ya en marcha; todos los clientes hablan con él."""
    return f"http://127.0.0.1:{PUERTO}/mcp"


def lanzar_http():
    """Lanza el servidor HTTP como un proceso y espera a que responda."""
    proceso = subprocess.Popen([sys.executable, str(SERVIDOR), str(DATOS), "--http", str(PUERTO)],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(100):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PUERTO}/mcp", timeout=1)
        except urllib.error.HTTPError:          # responde, aunque sea con un error: está listo
            return proceso
        except OSError:
            time.sleep(0.2)
    raise RuntimeError("El servidor HTTP no arranca.")


TRANSPORTES = {"en memoria": en_memoria, "stdio": por_stdio, "HTTP": por_http}


async def medir_transporte(nombre, llamadas=100):
    """Lo que tarda conectar (presentación y lista de herramientas) y cada llamada."""
    inicio = time.perf_counter()
    async with Client(TRANSPORTES[nombre]()) as c:
        await c.list_tools()
        conectar = time.perf_counter() - inicio
        inicio = time.perf_counter()
        for _ in range(llamadas):
            await c.call_tool("buscar_mascota", {"nombre": "Lola"})
        llamar = (time.perf_counter() - inicio) / llamadas
    return {"conectar_ms": round(conectar * 1000, 1), "llamada_ms": round(llamar * 1000, 2)}

## 3. Dos recepcionistas a la vez

In [ ]:
async def dos_recepcionistas(nombre, rondas=6):
    """Dos clientes con el mismo transporte. En cada ronda, los dos miran la primera hora
    libre del Dr. Menéndez el lunes 26 y los dos la reservan; si la reserva falla, el
    segundo vuelve a mirar y reserva la siguiente. Al final se cuentan las citas que
    acabaron a la misma hora con el mismo veterinario."""
    servidor_clinica.reiniciar()
    destino = TRANSPORTES[nombre]()
    reservas, rechazos = [], 0
    async with Client(destino) as ana, Client(destino) as luis:
        for ronda in range(rondas):
            huecos = []
            for c in (ana, luis):
                r = json.loads((await c.call_tool("huecos_libres", {
                    "fecha": "2026-10-26", "veterinario": "Menéndez"})).content[0].text)
                huecos.append(r["horas_libres"]["Dr. Pablo Menéndez"][0])
            for c, hora, mascota in ((ana, huecos[0], "Lola"), (luis, huecos[1], "Toby")):
                pedir = {"mascota": mascota, "veterinario": "Menéndez", "fecha": "2026-10-26",
                         "hora": hora, "motivo": f"revisión {ronda}"}
                r = await c.call_tool("reservar_cita", pedir)
                if r.is_error:
                    rechazos += 1
                    otra = json.loads((await c.call_tool("huecos_libres", {
                        "fecha": "2026-10-26", "veterinario": "Menéndez"})).content[0].text)
                    pedir["hora"] = otra["horas_libres"]["Dr. Pablo Menéndez"][0]
                    r = await c.call_tool("reservar_cita", pedir)
                if not r.is_error:
                    reservas.append(pedir["hora"])
    dobles = len(reservas) - len(set(reservas))
    return {"reservas": len(reservas), "rechazadas_y_repetidas": rechazos,
            "horas_reservadas_dos_veces": dobles}


async def experimento_transportes():
    servidor_clinica.cargar_datos(DATOS)
    proceso = lanzar_http()
    try:
        resultados = {}
        for nombre in TRANSPORTES:
            resultados[nombre] = {**await medir_transporte(nombre), **await dos_recepcionistas(nombre)}
            print(f"{nombre:12s}", resultados[nombre], flush=True)
    finally:
        proceso.terminate()
    return resultados

In [ ]:
transportes = await experimento_transportes()

## 4. Dos herramientas con el mismo nombre

Con el modelo. Dos servidores pequeños usan la misma palabra, `buscar`, para cosas distintas.

In [ ]:
tok, modelo = cargar(MODELO)

In [ ]:
SISTEMA = (
    "You are the assistant of the staff of Clínica Veterinaria Naranco, a veterinary "
    "clinic in Oviedo (Spain). Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in Spanish, in one or two sentences.")


def generar(mensajes, esquemas, max_tokens=400):
    """Una respuesta del modelo y los tokens de entrada y de salida."""
    texto = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                    add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    return (tok.decode(nuevos, skip_special_tokens=True).strip(),
            entrada["input_ids"].shape[1], len(nuevos))


def leer_llamadas(respuesta):
    """Las llamadas que pide el modelo, escritas entre <tool_call> y </tool_call>."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def dice(respuesta, *formas):
    """¿Aparece en la respuesta alguna de esas formas?"""
    return any(re.search(f, respuesta or "", re.I) for f in formas)


def sin_titulos(esquema):
    """El esquema sin las claves «title» que añade Pydantic y que el modelo no necesita."""
    if isinstance(esquema, dict):
        return {k: sin_titulos(v) for k, v in esquema.items() if k != "title"}
    return esquema


def compactar(texto):
    """El JSON de un resultado en una sola línea, sin la sangría con que lo envía el servidor."""
    try:
        return json.dumps(json.loads(texto), ensure_ascii=False)
    except ValueError:
        return texto


async def agente_mcp(tarea, anfitrion, max_pasos=8, ver=False):
    """El bucle del capítulo 1, con las herramientas de los servidores MCP."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta, entrada, salida = generar(mensajes, anfitrion.esquemas)
        llamadas = leer_llamadas(respuesta)[:1]
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "pensamiento": pensamiento,
                      "llamadas": []})
        if not llamadas:
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            texto = await anfitrion.llamar(llamada["name"], llamada["arguments"])
            traza[-1]["llamadas"].append({**llamada, "resultado": texto,
                                          "tokens_resultado": len(tok(texto)["input_ids"])})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?", "content": texto})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
                print(f"     → {texto[:150]}")
    return None, traza

In [ ]:
ficha = MCPServer("clinica")


@ficha.tool()
def buscar(nombre: Annotated[str, Field(description="Nombre de la mascota.")]) -> dict:
    """Busca una mascota de la clínica por su nombre: especie, raza, edad, peso y alergias."""
    return servidor_clinica.resumen_mascota(servidor_clinica.mascota_de(nombre))


farmacia = MCPServer("farmacia")


@farmacia.tool(name="buscar")
def buscar_medicamento(nombre: Annotated[str, Field(description="Nombre del medicamento.")]) -> dict:
    """Busca un medicamento de la farmacia: presentación, especies, dosis y existencias."""
    m = servidor_clinica.FARMACIA.get(servidor_clinica.sin_tildes(nombre))
    if m is None:
        raise ToolError(f"No tenemos «{nombre}».")
    return {"medicamento": servidor_clinica.sin_tildes(nombre), **{k: m[k] for k in (
        "presentacion", "especies", "dosis_mg_kg", "existencias")}}


class Anfitrion:
    """El anfitrión del capítulo 4 (compacto), con dos formas de reunir las herramientas:
    sin espacios de nombres, la última herramienta con un nombre sustituye a las anteriores;
    con espacios de nombres, cada herramienta se llama servidor_herramienta."""

    def __init__(self, servidores, espacios=False):
        self.servidores, self.espacios = servidores, espacios

    async def __aenter__(self):
        self.pila = AsyncExitStack()
        self.ruta, self.esquemas = {}, []
        for servidor in self.servidores:
            cliente = await self.pila.enter_async_context(Client(servidor))
            for h in (await cliente.list_tools()).tools:
                nombre = f"{cliente.server_info.name}_{h.name}" if self.espacios else h.name
                self.ruta[nombre] = (cliente, h.name)
                self.esquemas.append({"type": "function", "function": {
                    "name": nombre, "description": h.description,
                    "parameters": sin_titulos(h.input_schema)}})
        return self

    async def __aexit__(self, *error):
        await self.pila.aclose()

    async def llamar(self, nombre, argumentos):
        if nombre not in self.ruta:
            return json.dumps({"error": f"No existe la herramienta {nombre}."}, ensure_ascii=False)
        cliente, original = self.ruta[nombre]
        r = await cliente.call_tool(original, argumentos)
        return compactar("\n".join(b.text for b in r.content if b.type == "text"))


TAREAS = [
    ("raza_kira", "¿De qué raza es Kira?", lambda r: dice(r, "mestiz")),
    ("alergia_toby", "¿Toby es alérgico a algo?", lambda r: dice(r, "penicilina")),
    ("edad_zeus", "¿Cuántos años tiene Zeus?", lambda r: dice(r, r"\b4\b", "cuatro")),
    ("peso_max", "¿Cuánto pesa Max?", lambda r: dice(r, r"20[,.]7")),
    ("presentacion", "¿En qué presentación tenemos el meloxicam?", lambda r: dice(r, "suspensi")),
    ("especies", "¿En qué especies está indicado el maropitant?",
     lambda r: dice(r, "perro") and not dice(r, "gato")),
    ("existencias", "¿Cuántas unidades de enrofloxacino nos quedan?",
     lambda r: dice(r, rf"\b{servidor_clinica.FARMACIA['enrofloxacino']['existencias']}\b")),
    ("dosis_por_kilo", "¿Cuál es la dosis por kilo de la gabapentina?",
     lambda r: dice(r, rf"\b{servidor_clinica.FARMACIA['gabapentina']['dosis_mg_kg']:g}\b".replace(".", "[,.]"))),
]


async def experimento_nombres():
    servidor_clinica.cargar_datos(DATOS)
    resultados = {}
    for nombre, espacios in (("Sin espacios de nombres", False), ("Con espacios de nombres", True)):
        filas = []
        async with Anfitrion([ficha, farmacia], espacios) as anfitrion:
            for clave, tarea, comprobar in TAREAS:
                respuesta, traza = await agente_mcp(tarea, anfitrion)
                filas.append({"tarea": clave, "bien": bool(comprobar(respuesta)), "respuesta": respuesta,
                              "llamadas": [(l["name"], l["arguments"]) for p in traza for l in p["llamadas"]]})
            nombres = [e["function"]["name"] for e in anfitrion.esquemas]
        resultados[nombre] = {"bien": sum(f["bien"] for f in filas), "herramientas": nombres, "filas": filas}
        print(f"{nombre:26s}", resultados[nombre]["bien"], "de", len(TAREAS), nombres, flush=True)
        print("    " + " ".join(("+" if f["bien"] else "-") + f["tarea"] for f in filas), flush=True)
    return resultados

In [ ]:
nombres = await experimento_nombres()

## Ejercicios

1. Explica con tus palabras qué diferencia hay entre los tres transportes en dónde se ejecuta el servidor y en quién comparte sus datos.

2. Mide cuánto tarda en conectar el transporte stdio si el servidor no importa nada pesado. ¿De dónde salen los 1,5 segundos?

3. Repite la prueba de los dos recepcionistas con stdio, pero haciendo que los dos clientes usen el mismo proceso del servidor. ¿Se puede con el `Client` del SDK?

4. Añade una tercera recepcionista. ¿Cambia algo con HTTP? ¿Y con stdio?

5. Detén el servidor HTTP a mitad de la prueba y vuelve a lanzarlo. ¿Qué les pasa a las citas ya reservadas? ¿Qué haría falta para que no se perdieran?

6. Con el anfitrión sin espacios de nombres, imprime la lista de herramientas que recibe el modelo. ¿Cuántas `buscar` hay? ¿A qué servidor va cada llamada?

7. Cambia el anfitrión para que rechace al conectar una herramienta con un nombre ya usado, con un aviso. ¿Cuántas tareas hace bien el agente?

8. Prueba otra forma de espacio de nombres: dejar el nombre tal cual y añadir el servidor al principio de la descripción. ¿Basta para que el modelo distinga las dos herramientas?

9. Conecta el agente del capítulo 4 al servidor de la clínica por HTTP, en lugar de en memoria, y ejecuta las tareas del capítulo 1 que cubren sus herramientas. ¿Cambian los resultados?

10. **Proyecto.** Escribe una recomendación de una página para la clínica: qué transporte usar para cada servidor (la agenda, la farmacia del proveedor y la facturación) y por qué, con las cifras del capítulo.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Transportes de MCP](https://modelcontextprotocol.io/specification) en la especificación y [SDK de Python](https://github.com/modelcontextprotocol/python-sdk). Lecciones 13.08 y 13.09 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La Clínica Veterinaria Naranco y sus datos son ficticios. Código y experimentos propios.